# JARVIS Hybrid AI Training Manager

This notebook provides real, staged controls for hardware audit, Drive setup, data validation, baseline inference, LoRA training, evaluation, and release-manifest creation. It does not claim a model is trained until a real Colab GPU run completes. Keep private datasets, checkpoints, and adapters in Google Drive. Read ai-training/DATASET_GOVERNANCE.md before importing sources.

## Choose a stage

Run this notebook from the repository's ai-training/hybrid-brain-implementation branch. Run the cells in order. Begin with audit. A GPU is optional for data checks and required for model loading and training. A free Colab GPU is not guaranteed.

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
STAGE = 'audit'  # audit | validate | baseline | train | evaluate | release
MODEL_ID = ''    # choose only after reading model_candidates.json and checking license
MODEL_REVISION = ''  # immutable Hugging Face commit SHA; never use main
REPO_URL = 'https://github.com/18110priyanshuraj-creator/Jarvis.git'
REPO_DIR = Path('/content/Jarvis')
BRANCH = 'ai-training/hybrid-brain-implementation'
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO_DIR)],check=True)
else:
    current = subprocess.run(['git','-C',str(REPO_DIR),'branch','--show-current'],check=True,capture_output=True,text=True).stdout.strip()
    assert current == BRANCH, f'Existing checkout is on {current!r}, expected {BRANCH!r}; use a fresh Colab runtime.'
ROOT = REPO_DIR
os.chdir(ROOT)
assert (ROOT / 'ai-training' / 'scripts' / 'dataset_factory.py').exists(), 'Repository checkout is incomplete.'
print('Stage:', STAGE, '| Python:', sys.version.split()[0], '| platform:', platform.platform(), '| repo:', ROOT, '| branch:', BRANCH)

## Stage 1 — Hardware and persistent Drive setup

The hardware report records the assigned runtime. Drive mounting requires Google's normal authorization prompt. This stage creates the requested folder tree and writes a temporary test file to confirm Drive access.

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/JARVIS_AI')
folders = ['source_manifests','approved_corpus','derived_notes','datasets/general','datasets/editing','datasets/vision','datasets/audio','datasets/train','datasets/validation','datasets/test','runs/logs','runs/metrics','runs/hardware_reports','checkpoints/general','checkpoints/editing','checkpoints/combined','adapters/general','adapters/editing','adapters/combined','evaluation_reports','exports','backups']
for folder in folders: (DRIVE/folder).mkdir(parents=True,exist_ok=True)
cuda = torch.cuda.is_available()
gpu = torch.cuda.get_device_name(0) if cuda else None
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda else 0
ram_gb = round(__import__('psutil').virtual_memory().total / 1024**3,2) if __import__('importlib').util.find_spec('psutil') else None
report = {'created_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(), 'python': sys.version, 'pytorch': torch.__version__, 'cuda_available': cuda, 'gpu_name': gpu, 'gpu_vram_gb': vram_gb, 'ram_gb': ram_gb, 'disk_free_gb': round(shutil.disk_usage('/content').free/1024**3,2)}
print(json.dumps(report, indent=2))
if STAGE == 'audit':
    (DRIVE/'runs/hardware_reports/latest.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    probe=DRIVE/'backups'/'drive_write_probe.txt'; probe.write_text('JARVIS Drive write check',encoding='utf-8'); assert probe.read_text(encoding='utf-8')=='JARVIS Drive write check'; probe.unlink()
    print('Drive folders created and write/read check passed:',DRIVE)

## Stage 2 — Dependencies and dataset checks

Use the small authored sample only to check tooling. It is not enough data to train a useful model and it must not be presented as model-quality evidence. For a split, provide at least three independent source groups; whole source groups stay together.

In [ ]:
if STAGE in {'validate','baseline','train','evaluate','release'}:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','ai-training/requirements-colab.txt'],check=True)
if STAGE == 'validate':
    sample=ROOT/'ai-training/datasets/starter/general.jsonl'
    subprocess.run([sys.executable,'ai-training/scripts/dataset_factory.py','validate',str(sample)],check=True)
    print('For production: curate reviewed JSONL, validate sources, then split by source_group into Drive train/validation/test.')

## Stage 3 — Baseline model loading and inference

Select a candidate only after the hardware report exists and the license terms have been reviewed. The loader requires CUDA, at least 8 GiB VRAM, a pinned immutable revision, and a working chat template. The first actual run should record model ID and SHA in a run manifest.

In [ ]:
if STAGE == 'baseline':
    assert MODEL_ID and MODEL_REVISION, 'Set MODEL_ID and immutable MODEL_REVISION after the audit.'
    sys.path.insert(0,str(ROOT/'ai-training/scripts'))
    from model_runtime import load_model, generate
    tokenizer, model = load_model(MODEL_ID,MODEL_REVISION)
    messages=[{'role':'user','content':'Return a JSON edit plan for fading scene scene-2. Include a summary, operations, and notes.'}]
    print(generate(tokenizer,model,messages,max_new_tokens=160))

## Stage 4 — QLoRA training and recovery

Use reviewed, source-separated data in Drive. Training is refused without a supported GPU and immutable model revision. Checkpoints are written periodically and resume uses the latest saved checkpoint. Training will not run from the tiny sample.

In [ ]:
if STAGE == 'train':
    assert cuda and vram_gb >= 8, 'Training unavailable: an eligible CUDA GPU with at least 8 GiB VRAM is required.'
    assert MODEL_ID and MODEL_REVISION and MODEL_REVISION != 'main', 'Set a model ID and immutable SHA.'
    train_file=DRIVE/'datasets/train/train.jsonl'; valid_file=DRIVE/'datasets/validation/validation.jsonl'; run_dir=DRIVE/'runs/current'
    assert train_file.exists() and valid_file.exists(), 'Prepare reviewed train and validation JSONL files in Drive first.'
    cmd=[sys.executable,'ai-training/scripts/train_lora.py','--model',MODEL_ID,'--revision',MODEL_REVISION,'--train',str(train_file),'--validation',str(valid_file),'--output',str(run_dir)]
    if (run_dir/'checkpoints').exists() and any((run_dir/'checkpoints').glob('checkpoint-*')): cmd.append('--resume')
    subprocess.run(cmd,check=True)

## Stage 5 — Held-out evaluation

Keep evaluation examples separate from training and validation. The runner saves each answer and JSON-validity rate. This is a first measurement, not sufficient evidence for deployment; compare base and adapter on the same held-out set and inspect operation IDs and boundaries using the authoritative server validator.

In [ ]:
if STAGE == 'evaluate':
    assert cuda and MODEL_ID and MODEL_REVISION
    test_file=DRIVE/'datasets/test/test.jsonl'; assert test_file.exists(), 'Held-out test JSONL is missing.'
    base_report=DRIVE/'evaluation_reports/base.json'; adapter_report=DRIVE/'evaluation_reports/adapter.json'
    subprocess.run([sys.executable,'ai-training/scripts/evaluate.py','--model',MODEL_ID,'--revision',MODEL_REVISION,'--test',str(test_file),'--out',str(base_report)],check=True)
    adapter=DRIVE/'runs/current/adapter'; assert adapter.exists(), 'Trained adapter is missing.'
    subprocess.run([sys.executable,'ai-training/scripts/evaluate.py','--model',MODEL_ID,'--revision',MODEL_REVISION,'--test',str(test_file),'--adapter',str(adapter),'--out',str(adapter_report)],check=True)

## Stage 6 — Release manifest

Creating a manifest is not the same as approving or deploying a model. The release stays disabled until reviewed evaluation and a staging integration test pass. Never upload weights or private data to GitHub.

In [ ]:
if STAGE == 'release':
    import hashlib
    adapter=DRIVE/'runs/current/adapter'; assert adapter.exists(), 'No adapter exists; nothing to release.'
    files=sorted(p for p in adapter.rglob('*') if p.is_file())
    manifest={'status':'not-approved','deployment_enabled':False,'model_id':MODEL_ID,'revision':MODEL_REVISION,'adapter_files':[{'path':str(p.relative_to(adapter)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in files]}
    out=DRIVE/'exports'/'release-manifest.json'; out.write_text(json.dumps(manifest,indent=2),encoding='utf-8'); print(out); print(json.dumps(manifest,indent=2))

## Recovery and truthful status

Colab sessions can disconnect. Drive retains files; the temporary runtime does not. Resume with the same run directory and --resume. Verify the adapter directory, training_result.json, checkpoint files, and evaluation report before reporting success. No GPU training, checkpoint, model quality, or deployment is claimed by this notebook until those artifacts exist.